# Demo 3: Continuous Lebesgue Space in 3D

This notebook introduces `Lebesgue3D` as the continuous $L^2(\Omega)$ space on a box domain in **pygeoinf3D**.
The focus here is deliberately purely continuous: inner products, norms, orthogonality, and geometric identities in $L^2$, without any finite basis, components, or representation machinery.

**Sections:**
1. Imports and Setup
2. Defining a Continuous `Lebesgue3D` Space
3. Inner Products and Norms
4. Bilinearity and Symmetry
5. Orthogonality and Pythagorean Identity
6. Distance in $L^2$ and Slice Visualisation
7. Feature Summary

---
> **Visualisation mode:** in the visualisation section you can choose `"static"`, `"interactive"`, or `"both"` and re-run the last plotting cells.

## 1. Imports and Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pygeoinf3d import BoxDomain, Lebesgue3D, Function3D
from pygeoinf3d.plot import plot_function3d_axis_slice

box = BoxDomain(np.zeros(3), np.ones(3))
space = Lebesgue3D(box)

bbox = box.bounding_box()
print('Box domain:', box)
print('Low corner :', bbox.low)
print('High corner:', bbox.high)
print('Extents    :', box.extents)
print('Lebesgue space constructed on the unit cube.')

## 2. Defining a Continuous `Lebesgue3D` Space

`Lebesgue3D(box)` models the continuous Hilbert space $L^2(\Omega)$ over the domain $\Omega$.
In this notebook we treat it only through its continuous operations:

$$
\langle f, g \rangle_{L^2} = \int_\Omega f(\mathbf{x}) g(\mathbf{x})\,d\mathbf{x},
\qquad
\|f\|_{L^2} = \sqrt{\langle f, f \rangle}.
$$

We work with three vectorised functions on $[0,1]^3$:

$$
f(\mathbf{x}) = \sin(\pi x_1),
\qquad
g(\mathbf{x}) = \cos(\pi x_1),
\qquad
h(\mathbf{x}) = x_1 + x_2 + x_3.
$$

Here $f$ and $g$ are orthogonal on $[0,1]$, while $h$ is a simple non-oscillatory comparison field.

In [ ]:
def _as_point_batch(pts):
    pts = np.asarray(pts, dtype=float)
    is_single_point = pts.ndim == 1
    pts2 = np.atleast_2d(pts)
    return pts2, is_single_point

def f_callable(pts):
    pts2, is_single_point = _as_point_batch(pts)
    values = np.sin(np.pi * pts2[:, 0])
    return values[0] if is_single_point else values

def g_callable(pts):
    pts2, is_single_point = _as_point_batch(pts)
    values = np.cos(np.pi * pts2[:, 0])
    return values[0] if is_single_point else values

def h_callable(pts):
    pts2, is_single_point = _as_point_batch(pts)
    values = pts2[:, 0] + pts2[:, 1] + pts2[:, 2]
    return values[0] if is_single_point else values

f = Function3D(box, evaluate_callable=f_callable, name='f')
g = Function3D(box, evaluate_callable=g_callable, name='g')
h = Function3D(box, evaluate_callable=h_callable, name='h')

print('Created three continuous Function3D objects on the same domain:')
for fn in [f, g, h]:
    print(f'  {fn.name}: type={type(fn).__name__}')

pt = np.array([0.25, 0.50, 0.75])
print(f'\nSample point: {pt}')
print(f'f(pt) = {f(pt):.6f}')
print(f'g(pt) = {g(pt):.6f}')
print(f'h(pt) = {h(pt):.6f}')

## 3. Inner Products and Norms

The continuous $L^2$ inner product is computed numerically by quadrature over the box.
For the functions above we know:

$$
\langle f, f \rangle = \int_0^1 \sin^2(\pi x_1)\,dx_1 = \tfrac12,
\qquad
\langle g, g \rangle = \int_0^1 \cos^2(\pi x_1)\,dx_1 = \tfrac12,
$$

and

$$
\langle f, g \rangle = \int_0^1 \sin(\pi x_1)\cos(\pi x_1)\,dx_1 = 0.
$$

This gives a clean check of orthogonality and of the relation $\|f\|^2 = \langle f,f\rangle$.

In [ ]:
ip_ff = space.inner_product(f, f)
ip_gg = space.inner_product(g, g)
ip_fg = space.inner_product(f, g)

norm_f = space.norm(f)
norm_g = space.norm(g)

print(f'<f, f> = {ip_ff:.8f}   (expected 0.5)')
print(f'<g, g> = {ip_gg:.8f}   (expected 0.5)')
print(f'<f, g> = {ip_fg:.3e}   (expected ~0)')
print()
print(f'||f|| = {norm_f:.8f}   (expected {np.sqrt(0.5):.8f})')
print(f'||g|| = {norm_g:.8f}   (expected {np.sqrt(0.5):.8f})')
print(f'||f||^2 = {norm_f**2:.8f}')

import numpy.testing as npt
npt.assert_allclose(ip_ff, 0.5, atol=5e-3)
npt.assert_allclose(ip_gg, 0.5, atol=5e-3)
npt.assert_allclose(ip_fg, 0.0, atol=5e-3)
npt.assert_allclose(norm_f**2, ip_ff, atol=5e-3)
print('\nContinuous inner-product checks passed.')

## 4. Bilinearity and Symmetry

A Hilbert-space inner product must satisfy:

$$
\langle a f + b g, h \rangle = a\langle f,h \rangle + b\langle g,h \rangle,
\qquad
\langle f, g \rangle = \langle g, f \rangle.
$$

We verify those identities numerically using $a=2$ and $b=-0.5$.

In [ ]:
a = 2.0
b = -0.5
combo = a * f + b * g

lhs = space.inner_product(combo, h)
rhs = a * space.inner_product(f, h) + b * space.inner_product(g, h)
sym_fg = space.inner_product(f, g)
sym_gf = space.inner_product(g, f)

print(f'<a f + b g, h> = {lhs:.8f}')
print(f'a<f,h> + b<g,h> = {rhs:.8f}')
print(f'<f,g> = {sym_fg:.8f}')
print(f'<g,f> = {sym_gf:.8f}')

import numpy.testing as npt
npt.assert_allclose(lhs, rhs, atol=5e-3)
npt.assert_allclose(sym_fg, sym_gf, atol=5e-3)
print('\nBilinearity and symmetry checks passed.')

## 5. Orthogonality and Pythagorean Identity

For orthogonal functions, the $L^2$ norm behaves exactly like Euclidean geometry:

$$
\langle f, g \rangle = 0
\quad \Longrightarrow \quad
\|f + g\|^2 = \|f\|^2 + \|g\|^2.
$$

Since our $f$ and $g$ are orthogonal, this gives a second consistency check on the continuous inner-product implementation.

In [ ]:
f_plus_g = f + g
norm_sum_sq = space.norm(f_plus_g) ** 2
rhs = space.norm(f) ** 2 + space.norm(g) ** 2

print(f'||f + g||^2 = {norm_sum_sq:.8f}')
print(f'||f||^2 + ||g||^2 = {rhs:.8f}')

import numpy.testing as npt
npt.assert_allclose(norm_sum_sq, rhs, atol=5e-3)
print('\nPythagorean identity check passed for orthogonal functions.')

In [ ]:
parallelogram_lhs = space.norm(f + h) ** 2 + space.norm(f - h) ** 2
parallelogram_rhs = 2 * space.norm(f) ** 2 + 2 * space.norm(h) ** 2

print(f'||f + h||^2 + ||f - h||^2 = {parallelogram_lhs:.8f}')
print(f'2||f||^2 + 2||h||^2       = {parallelogram_rhs:.8f}')

import numpy.testing as npt
npt.assert_allclose(parallelogram_lhs, parallelogram_rhs, atol=5e-3)
print('\nParallelogram identity check passed.')

In [ ]:
VISUALIZATION_MODE = "interactive"  # change to "interactive" or "both"
INTERACTIVE_AXIS = "z"
INTERACTIVE_N_SLICES = 31
INTERACTIVE_RESOLUTION = 70
INTERACTIVE_FUNCTION_NAMES = ["f", "g", "h", "h_minus_f"]

try:
    from pygeoinf3d.plot import plot_function3d_axis_slider
    INTERACTIVE_PLOTTING_AVAILABLE = True
except ImportError:
    plot_function3d_axis_slider = None
    INTERACTIVE_PLOTTING_AVAILABLE = False

valid_modes = {"static", "interactive", "both"}
if VISUALIZATION_MODE not in valid_modes:
    raise ValueError(
        f"VISUALIZATION_MODE must be one of {sorted(valid_modes)}, got {VISUALIZATION_MODE!r}"
    )

print(f"VISUALIZATION_MODE = {VISUALIZATION_MODE!r}")
print(f"Interactive plotting available: {INTERACTIVE_PLOTTING_AVAILABLE}")

## 6. Visualisation Mode

This notebook supports two plotting styles for the final slice-visualisation section:

- `"static"`: Matplotlib slice heatmaps embedded directly in the notebook
- `"interactive"`: Plotly slice sliders from `pygeoinf3d.plot`
- `"both"`: render both versions

The interactive option depends on Plotly. If it is not installed, the notebook will print a short message instead of failing.

## 7. Distance in $L^2$ and Slice Visualisation

The $L^2$ norm induces a natural distance:

$$
d(f,h) = \|f-h\|_{L^2}.
$$

This measures global discrepancy over the whole domain, not pointwise error at a single location.
To complement that scalar distance, we visualise $z=0.5$ slices of the three continuous functions and of the difference $h-f$.

The cell below respects `VISUALIZATION_MODE` and can show static Matplotlib heatmaps, interactive Plotly slice sliders, or both.

In [ ]:
d_fh = space.norm(f - h)
d_gh = space.norm(g - h)

print(f'd(f, h) = ||f - h|| = {d_fh:.8f}')
print(f'd(g, h) = ||g - h|| = {d_gh:.8f}')

h_minus_f = h - f
functions = [f, g, h, h_minus_f]
titles = [
    r'$f(x)=\sin(\pi x_1)$',
    r'$g(x)=\cos(\pi x_1)$',
    r'$h(x)=x_1+x_2+x_3$',
    r'$h-f$',
]
function_map = {
    'f': (f, r'$f(x)=\sin(\pi x_1)$'),
    'g': (g, r'$g(x)=\cos(\pi x_1)$'),
    'h': (h, r'$h(x)=x_1+x_2+x_3$'),
    'h_minus_f': (h_minus_f, r'$h-f$'),
}

selected_names = []
for name in INTERACTIVE_FUNCTION_NAMES:
    if name not in function_map:
        raise ValueError(
            f"Unknown function name {name!r}. Choose from {sorted(function_map)}."
        )
    selected_names.append(name)

if VISUALIZATION_MODE in {'static', 'both'}:
    fig, axes = plt.subplots(2, 2, figsize=(10, 8))
    for ax, fn, title in zip(axes.flat, functions, titles):
        plot_function3d_axis_slice(fn, axis='z', coord=0.5, ax=ax, show_plot=False)
        ax.set_title(title)

    plt.suptitle(r'Continuous $L^2(\Omega)$ examples shown on the slice $z=0.5$', y=1.01)
    plt.tight_layout()
    plt.show()
else:
    print(
        "Skipping static slice heatmaps because VISUALIZATION_MODE='interactive'. "
        "Set VISUALIZATION_MODE to 'static' or 'both' to render them."
    )

if VISUALIZATION_MODE in {'interactive', 'both'}:
    if not INTERACTIVE_PLOTTING_AVAILABLE:
        print(
            "Interactive plotting requested, but Plotly is not installed. "
            "Install pygeoinf3d[interactive] or plotly to enable sliders."
        )
    else:
        print(
            f"Rendering interactive {INTERACTIVE_AXIS}-axis sliders for: "
            f"{', '.join(selected_names)}"
        )
        for name in selected_names:
            fn, label = function_map[name]
            fig_slider = plot_function3d_axis_slider(
                fn,
                axis=INTERACTIVE_AXIS,
                n_slices=INTERACTIVE_N_SLICES,
                resolution=INTERACTIVE_RESOLUTION,
                show_plot=False,
                title=f"{label} — interactive {INTERACTIVE_AXIS}-slice slider",
            )
            fig_slider.show()

## 8. Feature Summary

This notebook treated `Lebesgue3D` purely as a continuous $L^2$ space over a 3D box.
No basis, coefficients, components, or finite-dimensional representations were used.

| Continuous concept | API used |
|---|---|
| Domain | `BoxDomain(low, high)` |
| Continuous $L^2$ space | `Lebesgue3D(box)` |
| Scalar fields | `Function3D(box, evaluate_callable=...)` |
| Inner product | `space.inner_product(f, g)` |
| Norm | `space.norm(f)` |
| Orthogonality check | `space.inner_product(f, g) \approx 0` |
| $L^2$ distance | `space.norm(f - h)` |
| Static visualisation | `plot_function3d_axis_slice(...)` |
| Interactive visualisation | `plot_function3d_axis_slider(...)` |

**Next:** [3.1_kernel_functionals_3d_demo.ipynb](pygeoinf3D/demos/3.1_kernel_functionals_3d_demo.ipynb)